# Futures - Quick Start Guide

This notebook demonstrates the core functionality of the futures backtesting framework.

In [ ]:
# Setup - run from project root
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set display options
pd.set_option('display.max_columns', 20)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Data Loading

First, let's fetch some historical data using Alpaca.

In [ ]:
from futures.data import AlpacaFetcher, Storage
from futures.data.fetcher import DataManager
from datetime import date, timedelta

# Initialize data manager
dm = DataManager()

# Define tickers and date range
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'SPY']
end_date = date.today()
start_date = end_date - timedelta(days=365*2)  # 2 years

# Fetch data (will cache in SQLite)
data = dm.get_multi(tickers, start_date, end_date)

print(f"Loaded data for {len(data)} tickers")
for ticker, df in data.items():
    print(f"  {ticker}: {len(df)} bars from {df.index[0].date()} to {df.index[-1].date()}")

## 2. Technical Indicators

Let's compute some indicators on our data.

In [ ]:
from futures.indicators import SMA, EMA, RSI, MACD, BollingerBands
from futures.indicators.base import IndicatorPipeline

# Single indicator
aapl = data['AAPL'].copy()
aapl['sma_20'] = SMA(period=20)(aapl)
aapl['sma_50'] = SMA(period=50)(aapl)
aapl['rsi_14'] = RSI(period=14)(aapl)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Price and SMAs
axes[0].plot(aapl.index, aapl['close'], label='Close', alpha=0.8)
axes[0].plot(aapl.index, aapl['sma_20'], label='SMA 20', alpha=0.8)
axes[0].plot(aapl.index, aapl['sma_50'], label='SMA 50', alpha=0.8)
axes[0].set_title('AAPL Price with Moving Averages')
axes[0].legend()

# RSI
axes[1].plot(aapl.index, aapl['rsi_14'], label='RSI 14', color='purple')
axes[1].axhline(70, color='red', linestyle='--', alpha=0.5)
axes[1].axhline(30, color='green', linestyle='--', alpha=0.5)
axes[1].set_title('RSI')
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 3. Simple Strategy Backtest

Let's backtest a simple SMA crossover strategy.

In [ ]:
from futures.strategies import SMACrossover
from futures.backtester import Backtester, BacktestResult

# Create strategy
strategy = SMACrossover(fast_period=20, slow_period=50)

# Create backtester
backtester = Backtester(
    strategy=strategy,
    initial_cash=100_000,
    position_size=0.2,  # 20% per position
    max_positions=5,
)

# Run backtest
result = backtester.run(data)

# Print summary
print(result.summary())

In [ ]:
# Plot equity curve
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Equity curve
axes[0].plot(result.equity_curve.index, result.equity_curve.values)
axes[0].set_title(f'Equity Curve - {strategy.name}')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].axhline(100_000, color='gray', linestyle='--', alpha=0.5)

# Drawdown
cumulative = (1 + result.daily_returns).cumprod()
running_max = cumulative.expanding().max()
drawdown = (cumulative - running_max) / running_max * 100

axes[1].fill_between(drawdown.index, drawdown.values, 0, alpha=0.5, color='red')
axes[1].set_title('Drawdown')
axes[1].set_ylabel('Drawdown (%)')

plt.tight_layout()
plt.show()

## 4. Compare Multiple Strategies

In [ ]:
from futures.strategies import MeanReversion, MomentumStrategy
from futures.backtester.metrics import compare_strategies

# Define strategies to compare
strategies = [
    SMACrossover(fast_period=10, slow_period=30),
    SMACrossover(fast_period=20, slow_period=50),
    MeanReversion(bb_period=20, rsi_period=14),
    MomentumStrategy(),
]

# Run backtests
results = []
for strat in strategies:
    bt = Backtester(strategy=strat, initial_cash=100_000, position_size=0.2)
    result = bt.run(data, show_progress=False)
    results.append(result)
    print(f"{strat.name}: Return={result.total_return:.2f}%, Sharpe={result.metrics.sharpe_ratio:.2f}")

# Compare
comparison = compare_strategies(results)
comparison

## 5. ML Strategy

Now let's train an ML classifier and use it for trading.

In [ ]:
from futures.ml import FeatureEngineering, RandomForestModel, MLTrainer, WalkForwardValidator
from futures.strategies.ml_strategy import create_ml_strategy, MLStrategy

# Train ML model on historical data
# We'll use the first 80% for training
train_end = int(len(data['AAPL']) * 0.8)

train_data = {
    ticker: df.iloc[:train_end] 
    for ticker, df in data.items()
}

# Create ML strategy with SMA filter
ml_strategy = create_ml_strategy(
    train_data=train_data,
    model_class=RandomForestModel,
    use_sma_filter=True,
    sma_period=50,
    target_horizon=5,  # Predict 5-day returns
    n_estimators=100,
    max_depth=10,
)

print(f"Created strategy: {ml_strategy.name}")
print(f"Top features:")
for feat, imp in ml_strategy.model.get_top_features(10):
    print(f"  {feat}: {imp:.4f}")

In [ ]:
# Backtest ML strategy on test period
test_data = {
    ticker: df.iloc[train_end-60:]  # Include lookback for indicators
    for ticker, df in data.items()
}

ml_backtester = Backtester(
    strategy=ml_strategy,
    initial_cash=100_000,
    position_size=0.15,
    max_positions=5,
)

ml_result = ml_backtester.run(test_data)
print(ml_result.summary())

In [ ]:
# Walk-forward validation for a single stock
trainer = MLTrainer(
    model=RandomForestModel(n_estimators=50, max_depth=5),
)

wf_result = trainer.walk_forward_validate(
    df=data['AAPL'],
    target_horizon=5,
    train_window=252,
    test_window=21,
)

print(wf_result.summary())

## 6. Trade Analysis

In [ ]:
# Analyze trades from best performing strategy
best_result = max(results, key=lambda r: r.metrics.sharpe_ratio)
print(f"Best strategy: {best_result.strategy_name}")
print(f"\nTrade history:")
best_result.trades.head(10)

In [ ]:
# Trade P&L distribution
if not best_result.trades.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # P&L histogram
    axes[0].hist(best_result.trades['pnl'], bins=20, edgecolor='black')
    axes[0].axvline(0, color='red', linestyle='--')
    axes[0].set_title('Trade P&L Distribution')
    axes[0].set_xlabel('P&L ($)')
    
    # Win/Loss by ticker
    if 'ticker' in best_result.trades.columns:
        by_ticker = best_result.trades.groupby('ticker')['pnl'].sum().sort_values()
        colors = ['red' if x < 0 else 'green' for x in by_ticker.values]
        axes[1].barh(by_ticker.index, by_ticker.values, color=colors)
        axes[1].set_title('P&L by Ticker')
        axes[1].set_xlabel('Total P&L ($)')
    
    plt.tight_layout()
    plt.show()

## Next Steps

1. **Add more tickers** - Test on a broader universe
2. **Feature engineering** - Add cross-sectional features (relative strength)
3. **Hyperparameter tuning** - Optimize strategy parameters
4. **Risk management** - Add stop-loss, position sizing rules
5. **Sector analysis** - Group by sector, add sector rotation